In [21]:
%run common_setup.ipynb

In [22]:
class GiniExtractor(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def gini_citer_cited(self):

        sql_etl= """  
            WITH
            citer_cited_ETL AS
                (SELECT work_id AS citer_id,
                    unnest(referenced_works) AS cited_id
                FROM project.cited),
            authors_works_ETL AS
                (SELECT DISTINCT work_id, 
                        author_id
                FROM project.authorships
                WHERE author_id NOT NULL),
            citer_cited_authors_works_ETL AS
                (SELECT citer_id,
                        aw1.author_id AS citer_author,
                        cited_id,
                        aw2.author_id AS cited_author
                FROM citer_cited_ETL cc
                LEFT JOIN authors_works_ETL aw1
                ON aw1.work_id = cc.citer_id
                LEFT JOIN authors_works_ETL aw2
                ON aw2.work_id = cc.cited_id
                WHERE citer_author NOT NULL AND cited_author NOT NULL
                ORDER BY cited_author, citer_author),
            citer_ranking_ETL AS
                (SELECT cited_author,
                        citer_author,
                        count(citer_id) AS citer_count,
                        row_number() OVER (PARTITION BY citer_author ORDER BY count(citer_id) DESC) AS ranking
                FROM citer_cited_authors_works_ETL
                GROUP BY cited_author, citer_author)
            """

        sql = f"""
            CREATE OR REPLACE TABLE memory.gini_cited AS
                {sql_etl}    
                SELECT cited_author,
                        1.0-2.0*sum((citer_count*(ranking-1) + citer_count/2))/count(*)/sum(citer_count) AS gini_cited
                    FROM
                    (SELECT cited_author,
                            citer_author,
                            count(citer_id) AS citer_count,
                            row_number() OVER (PARTITION BY cited_author ORDER BY count(citer_id) DESC) AS ranking
                    FROM citer_cited_authors_works_ETL
                    GROUP BY cited_author, citer_author)
                    GROUP BY cited_author
                    HAVING sum(citer_count) > 0
                ORDER BY gini_cited DESC, cited_author
                """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.gini_cited").show()
        
        sql = f"""
            CREATE OR REPLACE TABLE memory.gini_citer AS
                {sql_etl}
                SELECT citer_author,
                        1.0-2.0*sum((cited_count*(ranking-1) + cited_count/2))/count(*)/sum(cited_count) AS gini_citer
                    FROM
                    (SELECT citer_author,
                            cited_author,
                            count(cited_id) AS cited_count,
                            row_number() OVER (PARTITION BY citer_author ORDER BY count(cited_id) DESC) AS ranking
                    FROM citer_cited_authors_works_ETL
                    GROUP BY citer_author, cited_author)
                    GROUP BY citer_author
                    HAVING sum(cited_count) > 0
                ORDER BY gini_citer DESC, citer_author
                """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.gini_citer").show()
        return

    def gini_coauthor(self):

        sql = """
            CREATE OR REPLACE TABLE memory.gini_coauthors AS  
                WITH 
                author_works_list_ETL AS
                    (SELECT author_id AS target_author,
                            list(work_id) AS works,
                            count(DISTINCT work_id) AS works_count
                    FROM (SELECT DISTINCT author_id, work_id
                            FROM project.authorships)
                    GROUP BY author_id),
                author_works_ETL AS
                        (SELECT target_author,
                            unnest(works) AS work_id
                        FROM author_works_list_ETL),
                coauthors_ETL AS
                    (SELECT DISTINCT target_author,
                            work_id,
                            author_id
                    FROM author_works_ETL
                        LEFT JOIN project.authorships
                        USING (work_id)
                        WHERE target_author != author_id),
                coauthor_counts_ETL AS
                    (SELECT target_author,
                            count(author_id) AS coauthor_instances,
                    FROM coauthors_ETL
                    GROUP BY target_author, author_id),
                ranking_ETL AS
                    (SELECT target_author,
                            coauthor_instances,
                            row_number() OVER (PARTITION BY target_author ORDER BY coauthor_instances DESC) AS ranking
                    FROM coauthor_counts_ETL
                    GROUP BY target_author, coauthor_instances
                    ORDER BY target_author, ranking)

                SELECT target_author,
                        sum(coauthor_instances) AS coauthors_unique,
                        1.0-2.0*sum((coauthor_instances*(ranking-1) + coauthor_instances/2))/count(*)/sum(coauthor_instances) AS gini_coauthors
                    FROM ranking_ETL
                    GROUP BY target_author
                    ORDER BY gini_coauthors DESC
                """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.gini_coauthors").show()
        return
    
    def combiner(self):
        sql = """ 
            CREATE OR REPLACE TABLE project.summary_full AS 
                SELECT s.*,
                        gini_citer, gini_cited, coauthors_unique, gini_coauthors, 
                        m.*, gc1.citer_author, gc2.cited_author, gc3.target_author
                    FROM project.summary s
                    LEFT JOIN memory.gini_citer gc1
                        ON s.author_id = gc1.citer_author
                        LEFT JOIN memory.gini_cited gc2
                            ON s.author_id = gc2.cited_author
                            LEFT JOIN memory.gini_coauthors gc3
                                ON s.author_id = gc3.target_author
                                LEFT JOIN project.sample_matched m
                                    ON s.author_id = m.author_id
                ORDER BY s."group" ASC, s.cited_by_count DESC
            """
        self.db.sql(sql)
        df = self.db.sql("SELECT * FROM project.summary_full").df()
        print(f'{df.shape = }\n{df.head()}\n{df.describe()}')
        with pd.ExcelWriter('../DATA/summary_full.xlsx') as writer:
            df.to_excel(writer, sheet_name='summary')
            df.describe().to_excel(writer, sheet_name='statistics')
            df[df['group'] != 'X'].describe().to_excel(writer, sheet_name='statistics_sample')
            df[df['group'] == 'C'].describe().to_excel(writer, sheet_name='statistics_sample_control')
            df[df['group'] == 'T'].describe().to_excel(writer, sheet_name='statistics_sample_target')
        return

In [23]:
def main():

    ge = GiniExtractor()
    ge.gini_citer_cited()
    ge.gini_coauthor()
    ge.combiner()
    ge.db.close()
    

In [24]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────────┬───────────┐
│ database │ schema  │         name         │     column_names     │             column_types              │ temporary │
│ varchar  │ varchar │       varchar        │      varchar[]       │               varchar[]               │  boolean  │
├──────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────────┼───────────┤
│ project  │ main    │ author_works_counts  │ [author_id, author…  │ [VARCHAR, VARCHAR, BIGINT, BIGINT, …  │ false     │
│ project  │ main    │ authors              │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ project  │ main    │ authorships          │ [work_id, author_i…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ project  │ main    │ authorships_full     │ [work_id, author_i…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ project  │ main    │ citation_